# VibeVoice-ASR Colab Notebook

Ang notebook na ito ay nagsasalin ng mga audio o video file.

## Mga Tampok

- **Libreng gamitin:** Ang notebook na ito ay gumagamit ng [VibeVoice-ASR](https://github.com/microsoft/VibeVoice), na open-source ng Microsoft noong Enero 21, 2026, kasama ng Google Colab na libreng GPU runtime. Upang gawing magagamit ang modelo sa mga libreng GPU, ang notebook na ito ay gumagamit ng quantized na VibeVoice-ASR na modelo.
- **Walang nakapirming limitasyon sa laki ng file sa gilid ng notebook:** Maaaring iproseso ang malalaking audio at video file sa pamamagitan ng paghahati sa mga ito sa mga segment.
- **Batch processing:** Maraming audio at video file ang maaaring iproseso sa isang pagtakbo.
- **Google Drive suporta:** Maaaring direktang i-upload ang mga file, o maaaring iproseso ang mga file na audio/video na nakaimbak sa Google Drive.
- **Ganap na online:** Ang mga pag-download at pagproseso ay isinasagawa sa Google Colab. Walang ginagamit na lokal na mapagkukunan ng computing.
- **Maraming format ng output:** Ang notebook ay naglalabas ng orihinal na JSON transcript at nagmula sa TXT, CSV, Markdown, Excel, at SRT na mga subtitle na file.

## Paano gamitin

1. Sa **1. Mga Setting**, piliin ang `INPUT_MODE`. I-edit ang iba pang mga setting kung kinakailangan.
2. Piliin ang **Rtime -> Run all**.
3. Kung `INPUT_MODE = "upload"`, maghintay hanggang **2. Magsisimula ang pagpili ng file / Google Drive mount**, pagkatapos ay i-click ang **Pumili ng Mga File** sa lugar ng output ng cell na iyon at i-upload ang iyong (mga) file.
4. Kapag kumpleto na ang pagproseso, awtomatikong dina-download ang mga na-transcribe na file.

## Mga Tala

- Ang Google Colab na libreng GPU ay may mga limitasyon sa paggamit. Pagkatapos ng isang tiyak na halaga ng paggamit, ang GPU access ay maaaring pansamantalang paghigpitan at ibalik sa ibang pagkakataon. Kung walang available na GPU, hihinto ang notebook na ito bago ang transkripsyon.
- Ang pagpoproseso ay tumatagal ng oras. Depende sa availability ng Colab, maaaring tumagal nang humigit-kumulang 10-20 minuto bago maging handa ang notebook na magproseso ng mga file. Bilang isang magaspang na gabay, ang isang 60 minutong audio o video file ay maaaring tumagal nang humigit-kumulang 1 oras upang ma-transcribe.
- Panatilihing bukas ang notebook na ito habang pinoproseso. Kung isasara mo ang notebook sa panahon ng transkripsyon, hihinto ang pagproseso.


In [ ]:
# @title 1. Mga setting
# ============================================================
# 1. Mga setting
# ============================================================

from pathlib import Path

# @markdown ## Input
# @markdown - `INPUT_MODE`: Piliin kung paano magbigay ng mga audio/video file. Gamitin ang `upload` para sa mga lokal na file, o `google_drive` para sa mga file sa Google Drive.
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]
# Pinapayagan ang maramihang mga file bilang default.
ALLOW_MULTIPLE_FILES = True

# @markdown ### Mga file sa Google Drive
# @markdown Ginagamit lang ang mga setting na ito kapag `INPUT_MODE == "google_drive"`.
# @markdown - `GOOGLE_DRIVE_ROOT`: Karaniwang panatilihin ang set na ito sa `/content/drive/MyDrive`.
# @markdown - `GOOGLE_DRIVE_SUBPATH`: Isang file o folder path sa ilalim ng MyDrive. Iwanang blangko upang gamitin ang top-level na folder ng iyong Google Drive MyDrive.
# @markdown - `RECURSIVE`: Kung ang `GOOGLE_DRIVE_SUBPATH` ay tumuturo sa isang folder, hanapin din ang mga subfolder nito.
# @markdown - `FILENAME_CONTAINS`: Opsyonal na filter ng substring ng filename. Iwanang blangko upang huwag paganahin ang pag-filter. Tumatanggap lamang ito ng isang string ng teksto; hindi nito sinusuportahan ang maramihang mga keyword o mga halagang pinaghihiwalay ng kuwit.
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

# Bumuo ng Drive input path mula sa root + relative subpath.
_mydrive_root = str(Path(GOOGLE_DRIVE_ROOT.strip().rstrip("/")))
_drive_subpath = GOOGLE_DRIVE_SUBPATH.strip().strip("/")
DRIVE_INPUT_PATH = str(Path(_mydrive_root) / _drive_subpath) if _drive_subpath else _mydrive_root

# @markdown ## Mga pagpipilian sa output
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: I-save ang final transcription output file sa Google Drive. Ito ay aktibo lamang kapag `INPUT_MODE == "google_drive"`.
# - `AUTO_DOWNLOAD_ZIP`: Isang ZIP file ang ginawa sa Colab. Kung nakatakda sa `False`, lalaktawan ang awtomatikong pag-download.
# - `SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE`: Kung nakatakda sa `True`, sine-save din ang mga checkpoint sa Google Drive. Hinahayaan ka nitong suriin ang mga bahagyang nabuong file kahit na hindi inaasahan ang pagpoproseso, ngunit nagiging sanhi ito ng madalas na pagsusulat sa Google Drive.
# - `GOOGLE_DRIVE_OUTPUT_ROOT`: Bilang default, ang isang subfolder para sa mga naka-save na output ay ginawa sa ilalim ng top-level na folder ng iyong Google Drive MyDrive.
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = True  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True
SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE = False
GOOGLE_DRIVE_OUTPUT_ROOT = str(Path(_mydrive_root) / "VibeVoice_ASR_outputs")

# ## Segmentasyon / henerasyon
# Ang audio ay panlabas na nahahati sa mga segment bago ang ASR. Ang mas malalaking segment ay nagpapanatili ng mas maraming konteksto ngunit nangangailangan ng mas maraming memorya at oras.
# Nililimitahan ng `MAX_NEW_TOKENS` ang nabuong transcript JSON bawat segment.
SEGMENT_SECONDS = 300
SEGMENT_OVERLAP_SECONDS = 5
MAX_NEW_TOKENS = 4096
# Panloob na acoustic tokenizer na nagpoproseso ng laki ng tipak sa mga sample ng audio, hindi mga segundo. Default 1440000 = 60 segundo sa 24 kHz.
ACOUSTIC_TOKENIZER_CHUNK_SIZE = 1440000

# @markdown ## Konteksto / Mga Hotword
# @markdown - `CONTEXT_INFO`: Opsyonal na konteksto na ginagamit bilang sanggunian sa panahon ng transkripsyon, gaya ng mga keyword, pangalan, organisasyon, at teknikal na termino. Iwanang blangko kung hindi kailangan. Maramihang mga keyword ay suportado; paghiwalayin ang mga ito gamit ang mga kuwit.
CONTEXT_INFO = ""  # @param {type:"string"}

# ## Mga setting ng henerasyon: deterministikong ASR-style na henerasyon.
# Kinokontrol ng mga parameter na ito ang pagbuo ng text pagkatapos ng audio tokenization. Ang mga default ay deterministic at ASR-oriented.
# - Ang `DO_SAMPLE=False` ay inirerekomenda para sa transkripsyon.
# - `TEMPERATURE=0.0` hindi pinapagana ang sampling-style na randomness.
# - Ang `TOP_P` ay may kaugnayan lamang kapag `DO_SAMPLE=True`.
# - Pinapanatili ng `NUM_BEAMS=1` ang henerasyon na simple at mahusay sa memorya.
# - Ang ibig sabihin ng `REPETITION_PENALTY=1.0` ay walang dagdag na parusa sa pag-uulit. Bahagyang itaas lamang kung lilitaw ang paulit-ulit na teksto.
DO_SAMPLE = False
TEMPERATURE = 0.0
TOP_P = 1.0
NUM_BEAMS = 1
REPETITION_PENALTY = 1.0
# Iba pang mga parameter: TOP_K, LENGTH_PENALTY, EARLY_STOPPING, NO_REPEAT_NGRAM_SIZE, EOS_TOKEN_ID, PAD_TOKEN_ID, USE_CACHE, RETURN_DICT_IN_GENERATE, OUTPUT_SCORES

# ## Mga advanced na setting ng modelo
# Ang mga advanced na setting ng modelo ay sadyang hindi bumubuo ng mga parameter para sa mga ordinaryong user.
# - BASE_MODEL_ID: HF Transformers bersyon ng VibeVoice-ASR ginamit bilang source model.
# - MODEL_LOAD_STRATEGY: "base_selective_nf4" load BASE_MODEL_ID na may selective NF4 quantization.
# - DUBEDO_MODEL_ID: ang pre-quantized na modelo ay itinatago lamang bilang isang sanggunian; ang direktang paglo-load ay naging sanhi ng mga isyu sa quantization ng acoustic tokenizer sa ilang kapaligiran.
# - ATTN_IMPLEMENTATION: gamitin ang "eager" dahil kasalukuyang hindi sinusuportahan ng VibeVoiceAcousticTokenizerEncoderModel ang SDPA sa Transformers path na ito.
# - INSTALL_FLASH_ATTENTION: maaaring magtagal o mabigo ang pag-install ng flash-attn sa Colab, kaya ang default ay False.
# - STREAM_OUTPUT: kapag True, ini-stream ang nabuong text sa lugar ng output habang pinoproseso ang bawat segment.
MODEL_LOAD_STRATEGY = "base_selective_nf4"
BASE_MODEL_ID = "microsoft/VibeVoice-ASR-HF"
DUBEDO_MODEL_ID = "Dubedo/VibeVoice-ASR-HF-NF4"
MODEL_PATH = BASE_MODEL_ID
ATTN_IMPLEMENTATION = "eager"
INSTALL_FLASH_ATTENTION = False
STREAM_OUTPUT = True

# Mga direktoryo ng runtime. Karaniwang hindi kailangang i-edit ng mga ordinaryong user ang mga ito.
UPLOAD_DIR = "/content/vibevoice_uploads"
WORK_DIR = "/content/vibevoice_work"
OUTPUT_DIR = "/content/vibevoice_outputs"

# Mga suportadong input media extension.
SUPPORTED_EXTS = {
    ".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".wma",
    ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"
}

# Mga module na dapat manatiling unquantized. Ang pagbibilang sa mga ito ay maaaring masira ang acoustic tokenizer path.
PROTECTED_MODULE_PREFIXES = [
    "acoustic_tokenizer_encoder",
    "semantic_tokenizer_encoder",
    "acoustic_projection",
    "semantic_projection",
    "lm_head",
]

# GPU-only notebook policy.
DEVICE = "cuda"
REQUIRE_CUDA = True
DTYPE_POLICY = "auto"  # auto uses bf16 when supported, otherwise fp16 on CUDA.
DEVICE_MAP_MODE = "single_gpu"
STREAM_PRINT_EVERY_CHARS = 1200

if int(ACOUSTIC_TOKENIZER_CHUNK_SIZE) % 3200 != 0:
    raise ValueError("ACOUSTIC_TOKENIZER_CHUNK_SIZE should be a multiple of 3200.")

print("Na-load ang mga setting.")
print(f"INPUT_MODE={INPUT_MODE}")
print(f"ALLOW_MULTIPLE_FILES={ALLOW_MULTIPLE_FILES}")
print(f"DRIVE_INPUT_PATH={DRIVE_INPUT_PATH}")
print(f"FILENAME_CONTAINS={FILENAME_CONTAINS}")
print(f"SAVE_OUTPUTS_TO_GOOGLE_DRIVE={SAVE_OUTPUTS_TO_GOOGLE_DRIVE}")
print(f"SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE={SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE}")
print(f"GOOGLE_DRIVE_OUTPUT_ROOT={GOOGLE_DRIVE_OUTPUT_ROOT}")
print(f"AUTO_DOWNLOAD_ZIP={AUTO_DOWNLOAD_ZIP}")
print(f"SEGMENT_SECONDS={SEGMENT_SECONDS}, SEGMENT_OVERLAP_SECONDS={SEGMENT_OVERLAP_SECONDS}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
print(f"ACOUSTIC_TOKENIZER_CHUNK_SIZE={ACOUSTIC_TOKENIZER_CHUNK_SIZE}")
print(f"MODEL_LOAD_STRATEGY={MODEL_LOAD_STRATEGY}, MODEL_PATH={MODEL_PATH}")
print(f"ATTN_IMPLEMENTATION={ATTN_IMPLEMENTATION}, DTYPE_POLICY={DTYPE_POLICY}")


In [ ]:
# @title 2. Pagpili ng file / Google Drive mount
# ============================================================
# 2. Pagpili ng file / Google Drive mount
# ============================================================
from pathlib import Path
import os
import shutil


Path(UPLOAD_DIR).mkdir(parents=True, exist_ok=True)
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def is_supported_media(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in SUPPORTED_EXTS


def collect_drive_files(path_str: str):
    root = Path(path_str)
    if root.is_file():
        if not is_supported_media(root):
            raise ValueError(f"Unsupported file type: {root}")
        return [str(root)]

    if not root.exists():
        raise FileNotFoundError(f"Drive path does not exist: {root}")

    pattern_iter = root.rglob("*") if RECURSIVE else root.glob("*")
    files = []
    for p in pattern_iter:
        if is_supported_media(p):
            if FILENAME_CONTAINS and FILENAME_CONTAINS not in p.name:
                continue
            files.append(str(p))
    files = sorted(files)
    return files


input_files = []

if INPUT_MODE.lower() == "upload":
    from google.colab import files
    from IPython.display import display, HTML
    display(HTML("""
    <div style='padding:10px 12px; border:1px solid #ddd; border-radius:8px; background:#fafafa; margin:8px 0;'>
      <b>Upload mode</b><br>
      Run this cell, then click <b>Choose Files</b> in the output area below.<br>
      If the code is hidden, the upload control should still appear as cell output unless the output itself is collapsed.
    </div>
    """))
    print("I-click ang 'Pumili ng Mga File' at mag-upload ng isa o higit pang mga audio/video na file.")
    uploaded = files.upload()
    for name in uploaded.keys():
        src = Path(name)
        dst = Path(UPLOAD_DIR) / src.name
        if src.exists():
            shutil.move(str(src), str(dst))
        if is_supported_media(dst):
            input_files.append(str(dst))
        else:
            print(f"Skipped unsupported file: {dst.name}")

elif INPUT_MODE.lower() == "google_drive":
    from google.colab import drive
    drive.mount('/content/drive')
    input_files = collect_drive_files(DRIVE_INPUT_PATH)
else:
    raise ValueError('INPUT_MODE must be "upload" or "google_drive".')

if not input_files:
    raise RuntimeError("Walang piniling mga suportadong audio/video file.")

print("Mga napiling file:")
for i, f in enumerate(input_files, 1):
    print(f"  {i}. {f}")

if len(input_files) > 1 and not ALLOW_MULTIPLE_FILES:
    raise RuntimeError(
        "Multiple files were found. To avoid accidental batch processing, either set DRIVE_INPUT_PATH to a single file, "
        "set FILENAME_CONTAINS, or set ALLOW_MULTIPLE_FILES=True."
    )


In [ ]:
# @title 3. GPU check
# ============================================================
# 3. GPU check
# ============================================================
import subprocess
import sys

try:
    import torch
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")
        print("BF16 supported:", torch.cuda.is_bf16_supported())
        try:
            print(subprocess.check_output(["nvidia-smi"], text=True))
        except Exception as e:
            print("nvidia-smi failed:", e)
    elif REQUIRE_CUDA:
        raise RuntimeError("Ang notebook na ito ay naka-configure na gamitin ang GPU. Sa Colab, piliin ang Runtime > Baguhin ang uri ng runtime > Hardware accelerator > GPU, pagkatapos ay i-restart at muling patakbuhin.")
except Exception as e:
    print("GPU check failed:", e)
    raise


In [ ]:
# @title 4. Mag-install ng mga dependency
# ============================================================
# 4. Mag-install ng mga dependency
# ============================================================
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def run_cmd(cmd, check=True):
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

# Huwag muling i-install ang sulo dito; gamitin ang CUDA-enabled na torch build ng Colab runtime.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run_cmd([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=5.3.0",
    "accelerate>=1.6.0",
    "bitsandbytes>=0.48.1",
    "huggingface_hub[hf_xet]",
    "librosa>=0.10.2",
    "soundfile>=0.12.1",
    "pandas>=2.0.0",
    "openpyxl>=3.1.0",
    "tqdm>=4.66.0",
    "pydub",
    "liger-kernel",
])

if INSTALL_FLASH_ATTENTION:
    # Maaaring mag-compile nang mahabang panahon o mabigo depende sa larawan ng Colab.
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "flash-attn", "--no-build-isolation"], check=False)

print("Natapos na ang hakbang sa pag-install ng dependency. Kung nabigo ang mga pag-import sa ibang pagkakataon, Runtime > I-restart ang runtime at patakbuhin muli ang mga cell.")


In [ ]:
# @title 5. I-load ang selective-NF4 quantized VibeVoice-ASR modelo
# ============================================================
# 5. I-load ang selective-NF4 quantized VibeVoice-ASR modelo
# ============================================================
import time
import json
import traceback
import threading
from pathlib import Path

import torch
from transformers import (
    AutoProcessor,
    VibeVoiceAsrForConditionalGeneration,
    BitsAndBytesConfig,
    TextIteratorStreamer,
    StoppingCriteria,
    StoppingCriteriaList,
)

# Opsyonal na Liger Kernel, tulad ng sa opisyal na demo. Nakakaapekto ito sa Qwen2 na mga bahagi kapag sinusuportahan ng runtime.
try:
    from liger_kernel.transformers import apply_liger_kernel_to_qwen2
    apply_liger_kernel_to_qwen2(
        rope=True,
        rms_norm=True,
        swiglu=True,
        cross_entropy=False,
    )
    print("✅ Liger Kernel applied to Qwen2 components")
except Exception as e:
    print(f"⚠️ Liger Kernel not applied: {e}")


def resolve_device():
    if DEVICE == "auto":
        resolved = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        resolved = DEVICE
    if REQUIRE_CUDA and resolved != "cuda":
        raise RuntimeError("This notebook is configured to use GPU. Set Colab runtime hardware accelerator to GPU.")
    if resolved == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable. Set Colab runtime hardware accelerator to GPU.")
    return resolved


def resolve_dtype(device: str):
    p = DTYPE_POLICY.lower()
    if p == "bf16":
        return torch.bfloat16
    if p == "fp16":
        return torch.float16
    if p == "fp32":
        return torch.float32
    # sasakyan
    if device == "cuda":
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32


def resolve_attn_implementation():
    # Sa HF selective-NF4 path na ito, kasalukuyang nabigo ang acoustic tokenizer encoder sa SDPA.
    # Samakatuwid, kahit na ang "auto" ay bumabalik sa "eager" maliban kung ang flash_attention_2 ay tahasang available at napili.
    if ATTN_IMPLEMENTATION != "auto":
        return ATTN_IMPLEMENTATION
    try:
        import flash_attn  # noqa: F401
        return "flash_attention_2"
    except Exception:
        return "eager"


def model_input_device(model):
    # Para sa device_map={"": 0}, ito ay cuda:0. Gumagana rin ito kung ang unang parameter ay nasa CUDA.
    for p in model.parameters():
        return p.device
    return torch.device("cuda:0")


stop_generation_flag = False

class StopOnFlag(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        global stop_generation_flag
        return stop_generation_flag


def normalize_hf_parsed(parsed):
    if parsed is None:
        return []
    if isinstance(parsed, list):
        if len(parsed) == 1 and isinstance(parsed[0], list):
            return parsed[0]
        if all(isinstance(x, dict) for x in parsed):
            return parsed
    if isinstance(parsed, dict):
        for key in ("segments", "transcription", "result"):
            if isinstance(parsed.get(key), list):
                return parsed[key]
    return []


class QuantizedVibeVoiceASRInference:
    """Wrapper with a v9-compatible transcribe() method, using HF selective NF4 quantization."""

    def __init__(self, model_path: str, device: str, dtype: torch.dtype, attn_implementation: str):
        self.device = device
        self.dtype = dtype
        self.attn_implementation = attn_implementation

        print(f"Loading processor: {model_path}")
        self.processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=False)

        if DEVICE_MAP_MODE == "single_gpu":
            device_map = {"": 0}
        else:
            device_map = "auto"

        print(f"Loading model with selective NF4 quantization: {model_path}")
        print(f"device_map={device_map}, compute dtype={dtype}, attn_implementation={attn_implementation}")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            # Sa kabila ng pangalan, ito ay ginagamit din ng BitsAndBytesConfig upang laktawan ang mga module sa panahon ng quantized loading.
            llm_int8_skip_modules=PROTECTED_MODULE_PREFIXES,
        )

        self.model = VibeVoiceAsrForConditionalGeneration.from_pretrained(
            model_path,
            device_map=device_map,
            dtype=dtype,
            quantization_config=bnb_config,
            attn_implementation=attn_implementation,
            trust_remote_code=False,
        )
        self.model.eval()
        self.input_device = model_input_device(self.model)

        print("✅ Quantized model loaded")
        print("hf_device_map:", getattr(self.model, "hf_device_map", None))
        try:
            print(f"input_device={self.input_device}, model dtype={self.model.dtype}")
        except Exception:
            print(f"input_device={self.input_device}")

        self._validate_quantization()

    def _validate_quantization(self):
        bad_rows = []
        bnb_count = 0
        for name, module in self.model.named_modules():
            cls_name = module.__class__.__name__
            cls_mod = module.__class__.__module__
            is_bnb = (
                "bitsandbytes" in cls_mod
                or "4bit" in cls_name.lower()
                or "8bit" in cls_name.lower()
                or "FP4" in cls_name
                or "NF4" in cls_name
            )
            if is_bnb:
                bnb_count += 1
                if any(name == p or name.startswith(p + ".") for p in PROTECTED_MODULE_PREFIXES):
                    bad_rows.append((name, cls_mod + "." + cls_name))

        print(f"bitsandbytes-like modules: {bnb_count}")
        if bad_rows:
            print("ERROR: Protected modules were quantized:")
            for name, cls in bad_rows[:40]:
                print(f" - {name}: {cls}")
            raise RuntimeError(
                "Protected acoustic/semantic tokenizer modules were quantized. "
                "Restart runtime and rerun with MODEL_LOAD_STRATEGY='base_selective_nf4'."
            )
        print("OK: protected acoustic/semantic tokenizer modules are not 4-bit quantized.")

    def _move_inputs_to_device(self, inputs):
        # Ang BatchEncoding.to(device, dtype) ay nagpapanatili ng integer tensor integer at nag-cast ng mga lumulutang na tensor.
        return inputs.to(self.input_device, self.dtype)

    def transcribe(self, audio_path: str, context_info: str = None, streamer=None) -> dict:
        prompt = context_info.strip() if context_info and context_info.strip() else None

        inputs = self.processor.apply_transcription_request(
            audio=str(audio_path),
            prompt=prompt,
        )
        inputs = self._move_inputs_to_device(inputs)

        generation_config = {
            "max_new_tokens": int(MAX_NEW_TOKENS),
            "acoustic_tokenizer_chunk_size": int(ACOUSTIC_TOKENIZER_CHUNK_SIZE),
            "do_sample": bool(DO_SAMPLE),
            "num_beams": int(NUM_BEAMS),
            "repetition_penalty": float(REPETITION_PENALTY),
            "stopping_criteria": StoppingCriteriaList([StopOnFlag()]),
        }
        if DO_SAMPLE:
            generation_config["temperature"] = float(TEMPERATURE)
            generation_config["top_p"] = float(TOP_P)
        if streamer is not None:
            generation_config["streamer"] = streamer

        input_token_count = int(inputs["input_ids"].shape[1]) if "input_ids" in inputs else None
        print(f"Input tokens: {input_token_count}")
        print("generate kwargs:", {k: v for k, v in generation_config.items() if k not in ("streamer", "stopping_criteria")})

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        start_time = time.time()
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, **generation_config)
        generation_time = time.time() - start_time

        generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
        output_token_count = int(generated_ids.shape[1])

        raw_text = ""
        segments = []
        transcription_only = ""

        try:
            raw_decoded = self.processor.decode(generated_ids)
            raw_text = raw_decoded[0] if isinstance(raw_decoded, list) else str(raw_decoded)
        except Exception as e:
            raw_text = f"[decode error] {repr(e)}"
            print("Warning: raw decode failed:", repr(e))

        try:
            parsed_decoded = self.processor.decode(generated_ids, return_format="parsed")
            parsed = parsed_decoded[0] if isinstance(parsed_decoded, list) and len(parsed_decoded) == 1 else parsed_decoded
            segments = normalize_hf_parsed(parsed)
        except Exception as e:
            print("Warning: parsed decode failed:", repr(e))
            segments = []

        try:
            transcription_decoded = self.processor.decode(generated_ids, return_format="transcription_only")
            transcription_only = transcription_decoded[0] if isinstance(transcription_decoded, list) else str(transcription_decoded)
        except Exception:
            transcription_only = ""

        try:
            del inputs, output_ids, generated_ids
            torch.cuda.empty_cache()
        except Exception:
            pass

        return {
            "raw_text": raw_text,
            "segments": segments,
            "transcription_only": transcription_only,
            "generation_time": generation_time,
            "input_tokens": {"total": input_token_count},
            "output_tokens": output_token_count,
        }


device = resolve_device()
dtype = resolve_dtype(device)
attn_impl = resolve_attn_implementation()

asr_model = QuantizedVibeVoiceASRInference(
    model_path=MODEL_PATH,
    device=device,
    dtype=dtype,
    attn_implementation=attn_impl,
)


In [ ]:
# @title 6. Mga function ng Helper: conversion, paghahati, pag-format, pag-save
# ============================================================
# 6. Mga function ng Helper: conversion, paghahati, pag-format, pag-save
# ============================================================
import os
import re
import math
import subprocess
import time
from pathlib import Path
from datetime import timedelta

import pandas as pd
from tqdm.auto import tqdm


def safe_stem(path: Path) -> str:
    s = path.stem
    s = re.sub(r"[\\/:*?\"<>|\s]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:160] or "audio"


def run_ffmpeg(args):
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"] + args
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr}")
    return result


def media_duration_seconds(path: Path) -> float:
    cmd = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path)
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed:\n{result.stderr}")
    return float(result.stdout.strip())


def convert_to_24k_mono_wav(input_path: Path, work_dir: Path) -> Path:
    out = work_dir / f"{safe_stem(input_path)}_24k_mono.wav"
    print(f"Converting to 24kHz mono WAV: {input_path}")
    run_ffmpeg(["-i", str(input_path), "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(out)])
    return out


def split_wav_for_asr(wav_path: Path, segment_seconds: int, overlap_seconds: int, work_dir: Path):
    duration = media_duration_seconds(wav_path)
    print(f"Duration: {duration:.1f} sec ({duration/60:.1f} min)")

    if duration <= segment_seconds:
        return [{"path": wav_path, "offset": 0.0, "duration": duration, "index": 1, "total": 1}]

    step = max(1, segment_seconds - overlap_seconds)
    starts = []
    t = 0.0
    while t < duration:
        starts.append(t)
        t += step

    segments = []
    total = len(starts)
    stem = safe_stem(wav_path)
    print(f"Splitting audio into {segment_seconds}s segments; overlap={overlap_seconds}s; total={total}")
    for idx, start in enumerate(starts, 1):
        seg_dur = min(segment_seconds, duration - start)
        if seg_dur <= 0:
            continue
        out = work_dir / f"{stem}_part{idx:03d}_{int(start):06d}s.wav"
        run_ffmpeg(["-ss", f"{start:.3f}", "-t", f"{seg_dur:.3f}", "-i", str(wav_path), "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(out)])
        segments.append({"path": out, "offset": float(start), "duration": float(seg_dur), "index": idx, "total": total})
    return segments


def normalize_segment(seg: dict, offset: float):
    # Ang opisyal na post_process_transcription ay nagbabalik ng mga susi gaya ng start_time/end_time/speaker_id/text.
    start = seg.get("start_time", seg.get("Start", seg.get("start", None)))
    end = seg.get("end_time", seg.get("End", seg.get("end", None)))
    speaker = seg.get("speaker_id", seg.get("Speaker", seg.get("speaker", None)))
    text = seg.get("text", seg.get("Content", seg.get("content", "")))
    if isinstance(start, (int, float)):
        start = float(start) + offset
    if isinstance(end, (int, float)):
        end = float(end) + offset
    return {
        "Start": start,
        "End": end,
        "Speaker": speaker,
        "Content": text,
        "SegmentOffsetSeconds": offset,
    }


def seconds_to_hms(x):
    if not isinstance(x, (int, float)):
        return ""
    x = max(0, float(x))
    h = int(x // 3600)
    m = int((x % 3600) // 60)
    s = x % 60
    return f"{h:02d}:{m:02d}:{s:06.3f}"


def seconds_to_srt_time(x):
    if not isinstance(x, (int, float)):
        x = 0.0
    x = max(0, float(x))
    h = int(x // 3600)
    m = int((x % 3600) // 60)
    s_int = int(x % 60)
    ms = int(round((x - int(x)) * 1000))
    if ms == 1000:
        s_int += 1
        ms = 0
    if s_int == 60:
        m += 1
        s_int = 0
    if m == 60:
        h += 1
        m = 0
    return f"{h:02d}:{m:02d}:{s_int:02d},{ms:03d}"


def rows_to_srt(rows):
    blocks = []
    seq = 1
    for r in rows:
        start = r.get("Start")
        end = r.get("End")
        if not isinstance(start, (int, float)):
            start = 0.0
        if not isinstance(end, (int, float)) or float(end) <= float(start):
            end = float(start) + 2.0

        text = str(r.get("Content", "")).strip()
        speaker = r.get("Speaker")
        if speaker is not None and str(speaker).strip() != "":
            text = f"[Speaker {speaker}] {text}".strip()

        blocks.append(
            f"{seq}\n"
            f"{seconds_to_srt_time(start)} --> {seconds_to_srt_time(end)}\n"
            f"{text}\n"
        )
        seq += 1
    return "\n".join(blocks).strip() + ("\n" if blocks else "")


def rows_to_txt(rows):
    lines = []
    for r in rows:
        t = f"[{seconds_to_hms(r.get('Start'))} - {seconds_to_hms(r.get('End'))}]"
        sp = r.get("Speaker")
        speaker = f" Speaker {sp}" if sp is not None else ""
        lines.append(f"{t}{speaker}\n{r.get('Content','')}\n")
    return "\n".join(lines)


def save_outputs(base_name: str, rows: list, raw_segments: list, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows)

    json_path = out_dir / f"{base_name}.json"
    txt_path = out_dir / f"{base_name}.txt"
    md_path = out_dir / f"{base_name}.md"
    csv_path = out_dir / f"{base_name}.csv"
    xlsx_path = out_dir / f"{base_name}.xlsx"
    srt_path = out_dir / f"{base_name}.srt"
    raw_path = out_dir / f"{base_name}_raw_segments.json"

    json_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    raw_path.write_text(json.dumps(raw_segments, ensure_ascii=False, indent=2), encoding="utf-8")
    txt_path.write_text(rows_to_txt(rows), encoding="utf-8")
    md_path.write_text("# Transcription\n\n" + rows_to_txt(rows), encoding="utf-8")
    srt_path.write_text(rows_to_srt(rows), encoding="utf-8")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df.to_excel(xlsx_path, index=False)

    return {
        "json": json_path,
        "raw_segments": raw_path,
        "txt": txt_path,
        "md": md_path,
        "csv": csv_path,
        "xlsx": xlsx_path,
        "srt": srt_path,
    }


def print_vram(prefix=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"{prefix} CUDA memory allocated={allocated:.2f} GiB, reserved={reserved:.2f} GiB")


In [ ]:
# @title 7. Pagpapatupad ng transkripsyon
# ============================================================
# 7. Pagpapatupad ng transkripsyon
# ============================================================
from pathlib import Path
from datetime import datetime
import threading
import time

work_dir = Path(WORK_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
all_results = []


def make_unique_dir(parent: Path, prefix: str, timestamp: str) -> Path:
    """Create a timestamped output directory and avoid collisions if the cell is rerun."""
    parent.mkdir(parents=True, exist_ok=True)
    candidate = parent / f"{prefix}_{timestamp}"
    if not candidate.exists():
        candidate.mkdir(parents=True, exist_ok=False)
        return candidate
    n = 2
    while True:
        candidate_n = parent / f"{prefix}_{timestamp}_{n:02d}"
        if not candidate_n.exists():
            candidate_n.mkdir(parents=True, exist_ok=False)
            return candidate_n
        n += 1


RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Mag-imbak ng mga intermediate checkpoint at mga huling output nang hiwalay.
# Ang ZIP download cell ay gumagamit lamang ng local outputs_* folder.
CHECKPOINT_OUTPUT_ROOT = make_unique_dir(output_dir, "checkpoints", RUN_TIMESTAMP)
FINAL_OUTPUT_ROOT = make_unique_dir(output_dir, "outputs", RUN_TIMESTAMP)

DRIVE_FINAL_OUTPUT_ROOT = None
DRIVE_CHECKPOINT_OUTPUT_ROOT = None
_google_drive_output_enabled = INPUT_MODE.lower() == "google_drive" and bool(SAVE_OUTPUTS_TO_GOOGLE_DRIVE)
_google_drive_checkpoint_enabled = INPUT_MODE.lower() == "google_drive" and bool(SAVE_CHECKPOINTS_TO_GOOGLE_DRIVE)

if _google_drive_output_enabled:
    DRIVE_FINAL_OUTPUT_ROOT = make_unique_dir(Path(GOOGLE_DRIVE_OUTPUT_ROOT), "outputs", RUN_TIMESTAMP)
if _google_drive_checkpoint_enabled:
    DRIVE_CHECKPOINT_OUTPUT_ROOT = make_unique_dir(Path(GOOGLE_DRIVE_OUTPUT_ROOT), "checkpoints", RUN_TIMESTAMP)

print(f"Local checkpoint folder: {CHECKPOINT_OUTPUT_ROOT}")
print(f"Local output folder: {FINAL_OUTPUT_ROOT}")
if DRIVE_FINAL_OUTPUT_ROOT:
    print(f"Google Drive output folder: {DRIVE_FINAL_OUTPUT_ROOT}")
if DRIVE_CHECKPOINT_OUTPUT_ROOT:
    print(f"Google Drive checkpoint folder: {DRIVE_CHECKPOINT_OUTPUT_ROOT}")


def transcribe_segment_official_style(seg_path: Path, context_info: str):
    """Transcribe one already-split WAV segment using the official-demo-style wrapper."""
    global stop_generation_flag
    stop_generation_flag = False

    if STREAM_OUTPUT:
        streamer = TextIteratorStreamer(
            asr_model.processor.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )
        result_container = {"result": None, "error": None}

        def run_job():
            try:
                result_container["result"] = asr_model.transcribe(
                    audio_path=str(seg_path),
                    context_info=context_info,
                    streamer=streamer,
                )
            except Exception as e:
                result_container["error"] = repr(e)
                traceback.print_exc()

        thread = threading.Thread(target=run_job)
        thread.start()
        generated_text_preview = ""
        last_print_len = 0
        start = time.time()
        for new_text in streamer:
            generated_text_preview += new_text
            if len(generated_text_preview) - last_print_len >= STREAM_PRINT_EVERY_CHARS:
                elapsed = time.time() - start
                print(f"  streaming: {len(generated_text_preview)} chars, {elapsed:.1f}s elapsed")
                print(generated_text_preview[last_print_len:len(generated_text_preview)])
                last_print_len = len(generated_text_preview)
        thread.join()
        if result_container["error"]:
            raise RuntimeError(result_container["error"])
        return result_container["result"]
    else:
        return asr_model.transcribe(
            audio_path=str(seg_path),
            context_info=context_info,
            streamer=None,
        )


for file_index, input_path_str in enumerate(tqdm(input_files, desc="Files"), start=1):
    input_path = Path(input_path_str)
    base_name = safe_stem(input_path)
    file_folder_name = f"{file_index:03d}_{base_name}"

    file_final_dir = FINAL_OUTPUT_ROOT / file_folder_name
    file_checkpoint_dir = CHECKPOINT_OUTPUT_ROOT / file_folder_name
    file_final_dir.mkdir(parents=True, exist_ok=True)
    file_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    drive_file_final_dir = DRIVE_FINAL_OUTPUT_ROOT / file_folder_name if DRIVE_FINAL_OUTPUT_ROOT else None
    drive_file_checkpoint_dir = DRIVE_CHECKPOINT_OUTPUT_ROOT / file_folder_name if DRIVE_CHECKPOINT_OUTPUT_ROOT else None
    if drive_file_final_dir:
        drive_file_final_dir.mkdir(parents=True, exist_ok=True)
    if drive_file_checkpoint_dir:
        drive_file_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Input: {input_path}")
    print(f"Local outputs for this file: {file_final_dir}")
    print(f"Local checkpoints for this file: {file_checkpoint_dir}")
    if drive_file_final_dir:
        print(f"Google Drive outputs for this file: {drive_file_final_dir}")
    if drive_file_checkpoint_dir:
        print(f"Google Drive checkpoints for this file: {drive_file_checkpoint_dir}")

    try:
        wav_path = convert_to_24k_mono_wav(input_path, work_dir)
        segments = split_wav_for_asr(wav_path, SEGMENT_SECONDS, SEGMENT_OVERLAP_SECONDS, work_dir)

        rows_all = []
        raw_segments_all = []
        file_start = time.time()

        for seg in segments:
            seg_path = seg["path"]
            offset = seg["offset"]
            idx = seg["index"]
            total = seg["total"]
            print("\n" + "-" * 80)
            print(f"Segment {idx}/{total} | offset={seconds_to_hms(offset)} | path={seg_path.name}")
            print_vram("Before segment")
            seg_start = time.time()

            result = transcribe_segment_official_style(seg_path, CONTEXT_INFO)
            seg_time = time.time() - seg_start

            print(f"Segment done: {seg_time:.2f}s | output_tokens={result.get('output_tokens')} | generation_time={result.get('generation_time'):.2f}s")
            print_vram("After segment")

            raw_segments_all.append({
                "segment_index": idx,
                "segment_total": total,
                "offset": offset,
                "duration": seg["duration"],
                "generation_time": result.get("generation_time"),
                "input_tokens": result.get("input_tokens"),
                "output_tokens": result.get("output_tokens"),
                "raw_text": result.get("raw_text", ""),
                "parsed_segments": result.get("segments", []),
            })

            parsed = result.get("segments", []) or []
            rows = [normalize_segment(s, offset) for s in parsed]
            if not rows:
                # Panatilihing nakikita ang hilaw na output kahit na nabigo ang structured parsing.
                rows = [{
                    "Start": offset,
                    "End": offset + seg["duration"],
                    "Speaker": None,
                    "Content": result.get("raw_text", ""),
                    "SegmentOffsetSeconds": offset,
                }]
            rows_all.extend(rows)

            # Mag-save ng checkpoint pagkatapos ng bawat segment sa hiwalay na folder ng checkpoint.
            checkpoint_paths = save_outputs(base_name + "_checkpoint", rows_all, raw_segments_all, file_checkpoint_dir)
            print(f"Checkpoint saved locally: {checkpoint_paths['json']}")
            if drive_file_checkpoint_dir:
                drive_checkpoint_paths = save_outputs(base_name + "_checkpoint", rows_all, raw_segments_all, drive_file_checkpoint_dir)
                print(f"Checkpoint saved to Google Drive: {drive_checkpoint_paths['json']}")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed = time.time() - file_start

        # I-save ang mga huling output nang lokal. Ang folder na ito ang dina-download ng ZIP na cell.
        paths = save_outputs(base_name, rows_all, raw_segments_all, file_final_dir)
        drive_paths = None
        if drive_file_final_dir:
            drive_paths = save_outputs(base_name, rows_all, raw_segments_all, drive_file_final_dir)

        print("\n--- ✅ Kumpleto na ang Transkripsyon ---")
        print(f"⏱️ Time: {elapsed:.2f}s | segments: {len(segments)} | rows: {len(rows_all)}")
        print("Local outputs:")
        for k, p in paths.items():
            print(f"  {k}: {p}")
        if drive_paths:
            print("Google Drive outputs:")
            for k, p in drive_paths.items():
                print(f"  {k}: {p}")

        all_results.append({
            "input": str(input_path),
            "local_output_dir": str(file_final_dir),
            "local_checkpoint_dir": str(file_checkpoint_dir),
            "google_drive_output_dir": str(drive_file_final_dir) if drive_file_final_dir else None,
            "google_drive_checkpoint_dir": str(drive_file_checkpoint_dir) if drive_file_checkpoint_dir else None,
            "outputs": {k: str(v) for k, v in paths.items()},
            "google_drive_outputs": {k: str(v) for k, v in drive_paths.items()} if drive_paths else None,
        })

    except Exception as e:
        print(f"\n❌ ERROR: {input_path}")
        print(repr(e))
        traceback.print_exc()

print("\n=== All done ===")
print(f"Local checkpoint folder: {CHECKPOINT_OUTPUT_ROOT}")
print(f"Local output folder: {FINAL_OUTPUT_ROOT}")
if DRIVE_FINAL_OUTPUT_ROOT:
    print(f"Google Drive output folder: {DRIVE_FINAL_OUTPUT_ROOT}")
if DRIVE_CHECKPOINT_OUTPUT_ROOT:
    print(f"Google Drive checkpoint folder: {DRIVE_CHECKPOINT_OUTPUT_ROOT}")
print(json.dumps(all_results, ensure_ascii=False, indent=2))


In [ ]:
# @title 8. I-zip at i-download ang mga final transcription output
# ============================================================
# 8. I-zip at i-download ang mga final transcription output
# ============================================================
# Lumilikha ang cell na ito ng ZIP archive sa Colab mula sa local outputs_* folder na ginawa ng cell 7.
# Ang ZIP file mismo ay hindi nakasulat sa Google Drive.
# Ang mga checkpoint file ay iniimbak sa isang hiwalay na timestamped na folder at hindi kasama.

from pathlib import Path
import zipfile

output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    raise FileNotFoundError(f"OUTPUT_DIR does not exist: {output_dir}")

# Mas gusto ang output folder na nilikha sa kasalukuyang runtime ng cell 7.
# Kung hindi available ang variable, bumalik sa pinakabagong outputs_* folder sa ilalim ng OUTPUT_DIR.
final_output_root_value = globals().get("FINAL_OUTPUT_ROOT", None)
if final_output_root_value is not None:
    final_output_root = Path(final_output_root_value)
else:
    candidates = sorted(output_dir.glob("outputs_*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise RuntimeError(f"No outputs_* folder found under: {output_dir}")
    final_output_root = candidates[0]

if not final_output_root.exists():
    raise FileNotFoundError(f"Final output folder does not exist: {final_output_root}")

files_to_zip = sorted(p for p in final_output_root.rglob("*") if p.is_file())
if not files_to_zip:
    raise RuntimeError(f"No files found under final output folder: {final_output_root}")

zip_name = f"{final_output_root.name}.zip"
zip_path = Path("/content") / zip_name

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        # Panatilihin ang mga output_* top-level na folder sa loob ng ZIP.
        zf.write(p, arcname=str(p.relative_to(final_output_root.parent)))

size_mb = zip_path.stat().st_size / 1024**2
print(f"Final output folder zipped: {final_output_root}")
print(f"Created ZIP on Colab: {zip_path}")
print(f"Files included: {len(files_to_zip)}")
print("Checkpoint files are not included because they are stored in a separate folder.")
print(f"ZIP size: {size_mb:.2f} MB")

if AUTO_DOWNLOAD_ZIP:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print("Automatic download was not started. Download the ZIP manually from the file browser:")
        print(zip_path)
        print(f"Reason: {e}")
else:
    print("AUTO_DOWNLOAD_ZIP is False. Download the ZIP manually from the Colab file browser if needed:")
    print(zip_path)
